# Targets: current share, remainder and the map they rebuild

Review notebook of the target phase (`docs/TARGET.md`). It reads the target tables, fitted objects and
build record of `scripts/build_targets.py`, the report of `scripts/profile_targets.py` (checks Y1 to
Y16) and the ceiling report of `scripts/current_ceiling.py`, and draws what the reports summarise.
Nothing here is fitted.

Run order: the ceiling, the target build and the profile come first; sections 1 to 7 are then
reviewed. Section 8 holds the headings of the findings, which are written to `docs/findings.md` at
step 7 of the procedure.

Cells are run manually, top to bottom. Section 2 reads the stored maps of one validation shot
(DIII-D, labelled). Section 4 reads the geometry columns of the first MAST public test shot.
Section 7 reads MAST test inputs from the frame table. No MAST equilibrium label is read anywhere.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fusion_eq import (coil_pattern as cp, current_basis as cb, frame_table as ft, split as sp, starter_kit,
                       target_build as tb, target_checks as tc, targets)
from fusion_eq.coil_flux import GreenTable, adopted_scales, element_outline
from fusion_eq.coil_scales import REFERENCE_PHASES
from fusion_eq.feature_checks import markdown_table
from fusion_eq.machines import machine_config

In [ ]:
DATA_ROOT = Path("../../data/raw/data")                     # stored maps of one validation shot, MAST coil geometry
TABLE_DIR = Path("../../data/processed/frame_table")        # frame table
TARGET_ROOT = Path("../../data/processed/targets")          # target tables, fitted objects and build.json
GREEN_DIR = Path("../../data/processed/green")              # grid tables
RESPONSE_CACHE = Path("../../data/processed/current_target")  # plasma responses
REPORT_ROOT = Path("../reports/targets")                    # target_report.md and target_summary.json
CEILING_PATH = Path("../reports/current_target/ceiling.json")
SPLIT_PATH = Path("../splits/split_v1.json")
CONFIG_PATH = Path("../configs/target_v1.yaml")
SPLIT_NAME = "diii_d_train"

ENVELOPE_FRAMES = 20_000    # reconstructed training shares for the envelope on MAST (as in the profile)
PLOT_FRAMES = 20_000        # validation frames in the ridge scatter
SEED = 0

In [ ]:
CONFIG = tb.TargetConfig.load(CONFIG_PATH)
FOLDER = tb.target_dir(CONFIG.version, TARGET_ROOT)
OBJECTS = tb.TargetObjects.load(FOLDER)
RECORD = OBJECTS.record
SUMMARY_PATH = REPORT_ROOT / f"v{CONFIG.version}" / "target_summary.json"
SUMMARY = json.loads(SUMMARY_PATH.read_text()) if SUMMARY_PATH.is_file() else {}
SPLIT = sp.read_split(SPLIT_PATH)
TABLES = {name: pd.read_parquet(FOLDER / f) for name, f in RECORD["tables"].items()}
FIT, VALIDATION, STRIDE = RECORD["fit_list"], RECORD.get("validation_list"), RECORD["stride"]
BASIS, REMAINDER = OBJECTS.current_basis, OBJECTS.remainder_model
GRIDS = {m: GreenTable.load(GREEN_DIR / f"green_{machine_config(m).file_prefix.split('_')[0]}.npz") for m in tb.MACHINES}
SCALES = {m: adopted_scales(m, GRIDS[m].columns) for m in tb.MACHINES}
RESPONSES = {m: cb.machine_response(m, GREEN_DIR, RESPONSE_CACHE) for m in tb.MACHINES}
C_COLUMNS = [f"c_{j:02d}" for j in range(BASIS.n_components)]
R_COLUMNS = [f"r_{j:02d}" for j in range(REMAINDER.n_components)]
RIDGE_COLUMNS = list(dict.fromkeys(c for r in REMAINDER.ridges.values() for c in r.columns))
SCORE_KEYS = ("S", "r2_psi", "r2_qb", "one_minus_dlcfs", "consistency", "r2_R_axis", "r2_Z_axis", "r2_kappa", "r2_tri_top",
              "r2_tri_bot", "r2_volume", "r2_li")
# Reference values of F79 (Y12 to Y15).
F79 = {"remainder_share": {"all": 0.0105, "rising": 0.0193, "flat_top": 0.0093, "late": 0.0143},
       "explained": {5: 0.9978, 10: 0.9998, 20: 1.0000}, "ridge": {"train": 0.8885, "validation": 0.8842},
       "f": {"S": 0.9982, "consistency": 0.9944, "r2_volume": 0.998}, "g": {"S": 0.9973, "consistency": 0.9911, "r2_volume": 0.991}}


def show(ax, grid, values, title, signed=True):
    """One map on a machine's grid in metres; signed maps on a diverging scale centred on zero."""
    v = np.asarray(values, dtype=np.float64).reshape(grid.grid_z.size, grid.grid_r.size)
    lim = float(np.nanmax(np.abs(v))) or 1.0
    kw = {"cmap": "RdBu_r", "vmin": -lim, "vmax": lim} if signed else {"cmap": "viridis"}
    im = ax.pcolormesh(grid.grid_r, grid.grid_z, v, shading="nearest", **kw)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=9)
    ax.set_xlabel("R (m)")
    ax.set_ylabel("Z (m)")
    plt.colorbar(im, ax=ax, fraction=0.046)


def draw_coils(ax, geometry):
    for i in range(geometry.n_elements):
        o = element_outline(geometry.r[i], geometry.z[i], geometry.width[i], geometry.height[i],
                            geometry.angle1[i], geometry.angle2[i])
        ax.fill(o[:, 0], o[:, 1], color="0.55", lw=0)


print(f"target build {FOLDER.name}: lists {list(TABLES)}, stride {STRIDE}, fit on {FIT}, validation {VALIDATION}")
if not SUMMARY:
    print(f"{SUMMARY_PATH} not found; run python scripts/profile_targets.py")

## 1. Build summary and the check table

The build record (configuration hash, machine constants and their verified status, the fit sample,
rows per list), the ring guard of Y9 and the check table of the profile. The Y9 table gives the
response against the filament Green's function beside the rule of F89.

In [ ]:
print(f"configuration hash at build {RECORD['config_hash'][:12]}, now {CONFIG.hash()[:12]}")
current = RECORD["current_basis"]
print(f"current basis: {current['n_components']} components ({current['even']} even, {current['odd']} odd), "
      f"variance held {current['explained_cumulative']:.6f}")
sample = RECORD["sample"]
print(f"fit sample: {sample['n_frames']} frames of {sample['requested']} requested (seed {sample['seed']}), "
      f"{sample['n_without_boundary']} drawn without a boundary and replaced")
print(f"remainder: {RECORD['remainder']['n_components']} components, default window {RECORD['remainder']['default_window']}; "
      "the 50 ms ridge keeps the toroidal field's 10 ms rate\n")
constants = [{"machine": m, "constant": k, "value": v["value"], "units": v["units"], "verified": v["verified"],
              "source": v.get("source", "")} for m, rec in RECORD["machine_constants"].items() for k, v in rec.items()]
print(markdown_table(pd.DataFrame(constants)), "\n")
print(markdown_table(pd.DataFrame([{"list": n, **r} for n, r in RECORD["lists"].items()]).fillna("")), "\n")
guard = sample.get("border_rings", {})
print("Ring guard: share of the fit sample's absolute current in the first interior rings\n")
print(markdown_table(pd.DataFrame([{"machine": m, **g} for m, g in guard.items()])), "\n")
if SUMMARY:
    print(tc.check_table(tc.Check(**c) for c in SUMMARY["checks"]), "\n")
    y9 = pd.DataFrame([{"machine": m, "beyond 5 cells": v["beyond_5"], "rule": v["thresholds"]["beyond_5"],
                        "beyond 10 cells": v["beyond_10"], "rule ": v["thresholds"]["beyond_10"],
                        f"symmetry, outside {v['symmetry_rings_excluded']} rings": v["symmetry"],
                        "rule  ": v["thresholds"]["symmetry"], "symmetry, all sources": v["symmetry_all_sources"],
                        "Z flip": v["flip"]} for m, v in SUMMARY["Y9"].items()])
    print(markdown_table(y9, digits=3))

## 2. One frame through the chain

The middle frame (at the build's stride) of the first validation shot: the stored map, the coil flux
under the adopted coil model, the current share inside the scorer's boundary, the flux of that
current (Ip G s), the remainder r, and the map recomposed from the three parts. The printed
differences are relative to the frame's flux range: the parts recompose the stored map exactly (Y2);
with the share through the basis and the ridge remainder, the difference is what a perfect
prediction of the coefficients would leave.

In [ ]:
FRAME_SHOT = sorted(SPLIT[VALIDATION or FIT])[0]
D3D, CFG_D3D = GRIDS["DIII-D"], machine_config("DIII-D")
shot_rows = ft.read_frame_table(SPLIT_NAME, ["shot", "frame", *targets.current_columns_of(D3D, "DIII-D"), "ip__mean", *RIDGE_COLUMNS],
                                shots=[FRAME_SHOT], root=TABLE_DIR).sort_values("frame").iloc[::STRIDE].reset_index(drop=True)
psi_shot = tb.read_truth(FRAME_SHOT, DATA_ROOT, SPLIT_NAME, STRIDE)["psi"]
k = len(shot_rows) // 2
ROW = shot_rows.iloc[[k]].reset_index(drop=True)
PSI = psi_shot[k:k + 1].astype(np.float64)
SHARE, rem_target = tb.split_frames(PSI, ROW, D3D, SCALES["DIII-D"], RESPONSES["DIII-D"], starter_kit.envelope_mask(CFG_D3D),
                                    starter_kit.scoring())
ip = ROW["ip__mean"].to_numpy(dtype=np.float64)
coil = targets.coil_flux_of_frames(D3D, ROW, SCALES["DIII-D"], CFG_D3D)
flux_j = ip[:, None, None] * 1e3 * cb.cell_flux(RESPONSES["DIII-D"], SHARE.share)
remainder = targets.sign_of_ip(ip)[:, None, None] * rem_target
recomposed = targets.to_stored(coil + flux_j + remainder, ip, CFG_D3D)
C_FRAME = BASIS.transform(SHARE.share)
predicted = cb.compose_map(CFG_D3D, coil, ip, C_FRAME, OBJECTS.flux_bases["DIII-D"], REMAINDER.predict(ROW))
span = float(PSI.max() - PSI.min())
print(f"shot {FRAME_SHOT}, frame {int(ROW['frame'].iloc[0])}, Ip {ip[0]:.0f} kA; share total {SHARE.total[0]:.4f}, "
      f"absolute share outside the boundary {SHARE.outside_abs_share[0]:.2e}")
print(f"recomposed from the parts against the stored map: {np.abs(recomposed - PSI).max() / span:.1e} of the flux range")
print(f"{BASIS.n_components} components and the ridge remainder ({REMAINDER.default_window}): "
      f"{np.abs(predicted - PSI).max() / span:.2e} of the flux range (largest), "
      f"{np.sqrt(np.mean((predicted - PSI) ** 2)) / span:.2e} (RMS)")
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
for ax, (values, title) in zip(axes.flat, ((PSI[0], "stored map (Wb/rad)"), (coil[0], "coil flux, physical frame"),
                                           (SHARE.share[0], "current share s"), (flux_j[0], "flux of the current, Ip G s"),
                                           (remainder[0], "remainder r"), (recomposed[0], "recomposed, stored frame"))):
    show(ax, D3D, values, title)
fig.tight_layout()

## 3. The mean share and the leading components

The mean share (even) and the leading components of the symmetric basis, with their parity and
their share of the variance (originals and mirrors). Then the section 2 frame and its mirror: the
coefficients of the flipped share equal the coefficients times the parity vector (Y5).

In [ ]:
MEAN = BASIS.mean.reshape(BASIS.shape)
ratio = BASIS.variances / BASIS.total_variance
n_show = min(7, BASIS.n_components)
fig, axes = plt.subplots(2, 4, figsize=(16, 9))
show(axes.flat[0], D3D, MEAN, "mean share", signed=False)
for j in range(n_show):
    show(axes.flat[j + 1], D3D, BASIS.components[j], f"component {j}: {'even' if BASIS.parity[j] > 0 else 'odd'}, "
                                                     f"{ratio[j]:.3f} of the variance")
for ax in axes.flat[n_show + 1:]:
    ax.axis("off")
fig.tight_layout()
print(f"{int((BASIS.parity > 0).sum())} even and {int((BASIS.parity < 0).sum())} odd components; variance held "
      f"{ratio.sum():.6f}")

In [ ]:
mirrored = cb.mirror_share(SHARE.share)
c_mirror = BASIS.transform(mirrored)
print(f"largest difference between the mirror's coefficients and c times p: "
      f"{np.abs(c_mirror - cb.mirror_coeffs(C_FRAME, BASIS.parity)).max():.1e}")
fig, axes = plt.subplots(1, 3, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 1, 1.6]})
show(axes[0], D3D, SHARE.share[0], "share of the frame")
show(axes[1], D3D, mirrored[0], "its mirror")
n_bar = min(20, BASIS.n_components)
x = np.arange(n_bar)
axes[2].bar(x - 0.2, C_FRAME[0, :n_bar], width=0.4, label="c")
axes[2].bar(x + 0.2, c_mirror[0, :n_bar], width=0.4, label="c of the mirror")
axes[2].plot(x, (BASIS.parity * C_FRAME[0])[:n_bar], "k_", ms=10, label="c times p")
axes[2].set_xlabel("component")
axes[2].legend(fontsize=8)
fig.tight_layout()

## 4. The current placed on MAST

The mean share and two leading components placed on the MAST grid with the constants of
`machines.py`, drawn in metres over the MAST coil elements and the grid (dashed). The black line is
the envelope of Y10: the cells inside the 99 percent support of at least 1 percent of the
reconstructed training shares (fixed-seed sample of `ENVELOPE_FRAMES`, as in the profile). The
markers are the centroids of the placed mean under the cases of Y11. No MAST label is read.

In [ ]:
MAST, PLACE = GRIDS["MAST"], OBJECTS.placements["MAST"]
geometry, _ = cp.geometry_from_split(DATA_ROOT, "MAST")
usable = TABLES[FIT][TABLES[FIT]["boundary_found"].to_numpy(dtype=bool) & TABLES[FIT]["passes_filter"].to_numpy(dtype=bool)]
pick = np.sort(np.random.default_rng(SEED + 1).permutation(len(usable))[:ENVELOPE_FRAMES])
shares = BASIS.inverse(usable.iloc[pick][C_COLUMNS].to_numpy(dtype=np.float64))
occupancy = np.zeros((MAST.grid_z.size, MAST.grid_r.size))
for start in range(0, len(shares), 2000):
    occupancy += tc.support(PLACE.apply(shares[start:start + 2000])).sum(axis=0)
ENVELOPE = occupancy / max(1, len(shares)) >= 0.01
placed_mean = PLACE.apply(MEAN)
odd = [j for j in range(BASIS.n_components) if BASIS.parity[j] < 0][:1]
shown = [(placed_mean, "placed mean share", False), (PLACE.apply(BASIS.components[0].reshape(BASIS.shape)), "component 0, placed", True)]
shown += [(PLACE.apply(BASIS.components[j].reshape(BASIS.shape)), f"component {j} (odd), placed", True) for j in odd]
fig, axes = plt.subplots(1, len(shown), figsize=(6 * len(shown), 8))
for ax, (values, title, signed) in zip(np.atleast_1d(axes), shown):
    show(ax, MAST, values, title, signed=signed)
    draw_coils(ax, geometry)
    ax.add_patch(plt.Rectangle((MAST.grid_r.min(), MAST.grid_z.min()), np.ptp(MAST.grid_r), np.ptp(MAST.grid_z),
                               fill=False, ls="--", color="0.3"))
    if ENVELOPE.any():
        ax.contour(MAST.grid_r, MAST.grid_z, ENVELOPE.astype(float), levels=[0.5], colors="k", linewidths=1)
cases = SUMMARY.get("Y10_Y11", {})
for name, v in cases.items():
    np.atleast_1d(axes)[0].plot(v["envelope"]["centroid_r_m"], v["envelope"]["centroid_z_m"], "o", ms=5, label=name)
if cases:
    np.atleast_1d(axes)[0].legend(fontsize=7, loc="lower right")
fig.tight_layout()
r0, a0 = machine_config("MAST").major_radius.value, machine_config("MAST").minor_radius.value
print(f"MAST constants R0 {r0} m, a0 {a0} m; {len(shares)} reconstructed training shares; envelope {int(ENVELOPE.sum())} cells; "
      f"mean share kept by placement {placed_mean.sum() / MEAN.sum():.12f}")

In [ ]:
rows = []
for name, v in SUMMARY.get("Y10_Y11", {}).items():
    e = v["envelope"]
    rows.append({"case": name, "R0 (m)": v["r0_m"], "a0 (m)": v["a0_m"], "centroid R": e["centroid_r_m"], "centroid Z": e["centroid_z_m"],
                 "inner R": e["inner_r_m"], "outer R": e["outer_r_m"], "height": e["height_m"], "nearest coil": e["coil_distance_m"],
                 "grid edge": e["edge_distance_m"], "coil elements inside": e.get("n_elements_inside", 0)})
print("Y10 and Y11, lengths in m\n")
print(markdown_table(pd.DataFrame(rows)) if rows else "no Y10 and Y11 in the profile summary")

## 5. Remainder

The leading components of the remainder basis (target frame, sign(Ip) r, Wb/rad), the remainder's RMS
over the frame's flux range by phase beside F79 (Y12), the variance held (Y13), and the ridge
prediction of the leading coefficients against the truth on the validation list (Y14).

In [ ]:
rb = REMAINDER.basis
n_show = min(4, REMAINDER.n_components)
fig, axes = plt.subplots(1, n_show + 1, figsize=(4.2 * (n_show + 1), 5))
show(axes[0], D3D, rb.mean, "remainder mean")
explained = np.diff(np.concatenate([[0.0], REMAINDER.explained_cumulative]))
for j in range(n_show):
    show(axes[j + 1], D3D, rb.components[j], f"component {j}, {explained[j]:.4f} of the variance")
fig.tight_layout()
held = [{"components": k, "held": float(REMAINDER.explained_cumulative[k - 1]), "F79": F79["explained"][k]}
        for k in (5, 10, 20) if k <= len(REMAINDER.explained_cumulative)]
print("Y13, variance held\n")
print(markdown_table(pd.DataFrame(held)))

In [ ]:
rows = []
for name, t in TABLES.items():
    found = t["boundary_found"].to_numpy(dtype=bool)
    for phase in ("all", *REFERENCE_PHASES):
        sel = found if phase == "all" else found & (t["phase"] == phase).to_numpy()
        q = tc.quantiles(t.loc[sel, "remainder_rms_over_range"])
        rows.append({"list": name, "phase": phase, "n": q["n"], "5th": q["q5"], "median": q["q50"], "95th": q["q95"],
                     "F79 median": F79["remainder_share"][phase]})
print("Y12, remainder RMS over the flux range\n")
print(markdown_table(pd.DataFrame(rows)))
fig, ax = plt.subplots(figsize=(8, 4.5))
fit = TABLES[FIT]
for phase in REFERENCE_PHASES:
    v = fit.loc[fit["boundary_found"].to_numpy(dtype=bool) & (fit["phase"] == phase).to_numpy(), "remainder_rms_over_range"]
    v = v[np.isfinite(v)]
    if len(v):
        ax.hist(v, bins=np.geomspace(max(v.min(), 1e-5), v.max() * 1.01, 60), histtype="step", label=phase)
ax.set_xscale("log")
ax.set_xlabel("remainder RMS over the flux range")
ax.set_ylabel("frames")
ax.set_title(f"{FIT}, by phase", fontsize=10)
ax.legend()

In [ ]:
name = VALIDATION or FIT
table = TABLES[name]
keep = table["passes_filter"].to_numpy(dtype=bool) & table["boundary_found"].to_numpy(dtype=bool)
frames = ft.read_frame_table(SPLIT_NAME, ["shot", "frame", *RIDGE_COLUMNS], shots=sorted(set(table["shot"].astype(str))), root=TABLE_DIR)
frames["shot"], frames["frame"] = frames["shot"].astype(str), frames["frame"].astype(np.int64)
truth = table.loc[keep, ["shot", "frame", *R_COLUMNS]].assign(shot=lambda d: d["shot"].astype(str), frame=lambda d: d["frame"].astype(np.int64))
joined = truth.merge(frames, on=["shot", "frame"], how="left", validate="one_to_one")
joined = joined.iloc[np.sort(np.random.default_rng(SEED).permutation(len(joined))[:PLOT_FRAMES])].reset_index(drop=True)
scores = [{"window": w, "training R2": r.scores.get("train", np.nan), "validation R2": r.scores.get("validation", np.nan),
           "default": w == REMAINDER.default_window} for w, r in REMAINDER.ridges.items()]
print("Y14, R2 of the remainder coefficients (F79: training 0.8885, validation 0.8842); the 50 ms ridge keeps the toroidal "
      "field's 10 ms rate\n")
print(markdown_table(pd.DataFrame(scores)))
pred = REMAINDER.ridges[REMAINDER.default_window].coefficients(joined)
n_show = min(3, REMAINDER.n_components)
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4.6))
for j, ax in enumerate(np.atleast_1d(axes)):
    t = joined[R_COLUMNS[j]].to_numpy(dtype=np.float64)
    ax.plot(t, pred[:, j], ".", ms=2, alpha=0.4)
    lo, hi = np.nanmin(t), np.nanmax(t)
    ax.plot([lo, hi], [lo, hi], "k-", lw=0.8)
    ax.set_xlabel(f"{R_COLUMNS[j]}, truth")
    ax.set_ylabel("ridge")
    ax.set_title(f"{name}, window {REMAINDER.default_window}", fontsize=9)
fig.tight_layout()

## 6. Ceiling variants

Variants h to k of `scripts/current_ceiling.py` beside f and g of F79 (Y15). Readings of
`docs/TARGET.md`, Part 5: h keeps 50 components if S is within 0.0003 of f and consistency within
0.001; i is a pessimistic estimate of the placement cost on MAST; j decides whether the predicted
share is rescaled to a total of one; k is the ceiling of the whole target chain on DIII-D.

In [ ]:
if CEILING_PATH.is_file():
    scores = json.loads(CEILING_PATH.read_text()).get("scores", {})
    rows = [{"variant": v, **{k: s.get(k, np.nan) for k in SCORE_KEYS}} for v, s in scores.items()
            if v[:2] in ("f_", "g_", "h_", "i_", "j_", "k_")]
    print(f"F79: f S {F79['f']['S']}, consistency {F79['f']['consistency']}; g S {F79['g']['S']}, consistency {F79['g']['consistency']}\n")
    print(markdown_table(pd.DataFrame(rows)) if rows else "the ceiling report has no variant f to k")
else:
    print(f"{CEILING_PATH} not found; run python scripts/current_ceiling.py")

## 7. Mean-current maps on both machines

`mean_current_v0`: the coil flux plus Ip phi_mean, with the ridge remainder on DIII-D and none on
MAST (Y16). The score table is that of the profile on DIII-D validation, beside `physics_mean` of F59
(R2 psi 0.991, S 0.800). The test-frame table gives, over the frames on which the scorer finds a closed
boundary, the derived R_axis, Z_axis and kappa and the boundary's inner and outer radius, width and
height in metres (median and 5th to 95th percentile). The maps are the section 2 frame (stored map,
mean current with and without the remainder) and the middle frame of the first MAST test shot, with
the boundary the scorer finds in the composed map.

In [ ]:
y16 = SUMMARY.get("Y16", {})
arms = [{"arm": k, **{x: s.get(x, np.nan) for x in SCORE_KEYS}} for k, s in y16.items() if k.startswith("validation")]
print("Y16, DIII-D validation (physics_mean of F59: R2 psi 0.991, S 0.800)\n")
print(markdown_table(pd.DataFrame(arms)) if arms else "no validation arm in the profile summary")
tests = {k[len("test_"):]: v for k, v in y16.items() if k.startswith("test_")}
print("\nY16, test frames (inputs only)\n")
if not tests:
    print("no test frames in the profile summary")
elif not all("spread" in v for v in tests.values()):
    print(markdown_table(pd.DataFrame([{"machine": m, **{x: y for x, y in v.items() if not isinstance(y, dict)}} for m, v in tests.items()])))
    print("\nthe profile summary has no boundary size; rerun python scripts/profile_targets.py")
else:
    rows = [{"machine": m, "frames": v["n_frames"], "boundary found": v["boundary_share"], "quantity": q,
             "median": s["q50"], "5th": s["q5"], "95th": s["q95"]} for m, v in tests.items() for q, s in v["spread"].items()]
    print(markdown_table(pd.DataFrame(rows)))

In [ ]:
with_r = tb.mean_current_maps("DIII-D", ROW, D3D, SCALES["DIII-D"], OBJECTS, with_remainder=True)
without_r = tb.mean_current_maps("DIII-D", ROW, D3D, SCALES["DIII-D"], OBJECTS, with_remainder=False)
mast_rows = ft.read_frame_table("mast_public_test", ["shot", "frame", *targets.current_columns_of(MAST, "MAST"), "ip__mean"], root=TABLE_DIR)
mast_rows["shot"] = mast_rows["shot"].astype(str)
mast_shot = sorted(mast_rows["shot"].unique())[0]
m_rows = mast_rows[mast_rows["shot"] == mast_shot].sort_values("frame")
m_row = m_rows.iloc[[len(m_rows) // 2]].reset_index(drop=True)
m_map = tb.mean_current_maps("MAST", m_row, MAST, SCALES["MAST"], OBJECTS)
scorer = starter_kit.scoring()
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for ax, (grid, values, title) in zip(axes, ((D3D, PSI[0], f"DIII-D {FRAME_SHOT}: stored map"),
                                            (D3D, with_r[0], "mean current, with the remainder"),
                                            (D3D, without_r[0], "mean current, without the remainder"),
                                            (MAST, m_map[0], f"MAST {mast_shot}: mean current"))):
    show(ax, grid, values, title)
env = starter_kit.envelope_mask(machine_config("MAST"))
contour = scorer.lcfs.extract_lcfs(m_map[0].astype(np.float64), env.grid_R, env.grid_Z, "MAST", env.mask_coarse, env.mask_f)
if contour is not None:
    axes[3].plot(contour[:, 0], contour[:, 1], "k-", lw=1)
span = float(PSI.max() - PSI.min())
print(f"DIII-D frame: RMS difference to the stored map over the flux range, with the remainder "
      f"{np.sqrt(np.mean((with_r - PSI) ** 2)) / span:.4f}, without {np.sqrt(np.mean((without_r - PSI) ** 2)) / span:.4f}")
print(f"MAST frame {int(m_row['frame'].iloc[0])}, Ip {m_row['ip__mean'].iloc[0]:.0f} kA: the scorer "
      f"{'finds' if contour is not None else 'finds no'} closed boundary in the composed map")
if contour is not None:
    print("boundary of that frame (m): " + ", ".join(f"{k} {v:.3f}" for k, v in tc.boundary_extent(contour).items()))
fig.tight_layout()

## 8. Findings

### Number of current components

50 symmetric components (30 even, 20 odd). Variant h at 50 is 0.0002 below f in S and 0.0006 in
consistency, inside the rule of Part 5 (0.0003 and 0.001). 80 components, the only count above f
(+0.0004 in S), is a configuration option (F90).

### Whether the model phase rescales the predicted share

No. Rescaling to a total of one (j) costs 0.0010 in S and 0.023 in volume; the total of the
reconstructed share runs from 0.989 to 1.007 (5th to 95th percentile) (F90).

### Placement resolution

The conservative remap is kept: the pessimistic estimate (i, two remaps) costs 0.0001 in S and
nothing in volume (F90).

### Rate window of the remainder ridge

50 ms. On the full build the validation R2 of the coefficients is 0.906 at 50 ms against 0.886 at
10 ms. The toroidal field keeps its 10 ms rate, the frame table having no 50 ms rate for it (F91).

### MAST reference radii

R0 0.85 m and a0 0.65 m (Sykes et al. 2001) stand. With them the placed current lies inside the grid,
0.109 m from the nearest coil element, its envelope reaching 1.47 m; the backup radii (0.70 m,
0.50 m) put the outer edge at 1.20 m. The envelope follows the constants by construction; the
boundary size on the MAST test frames (section 7) is the label-free check (F94, F95).

### Target B or the fallback

Target B stands. The whole chain on DIII-D (k at 50 ms) reaches S 0.9972, 0.0001 below g and 0.0018
below the plasma-flux basis (d); the fallback stays as defined on 12 October (F91).